# 01 — Entraînement ModCloth Fit Model V2

**But :** entraîner et évaluer le modèle TensorFlow/Keras de prédiction de fit (`small`, `fit`, `large`) à partir du dataset ModCloth.

Ce notebook exécute uniquement la V2 du pipeline ModCloth :
- téléchargement Kaggle ;
- inspection des données ;
- entraînement du MLP ;
- métriques et artefacts versionnés ;
- sauvegarde finale dans Google Drive.

Il ne traite ni Fashion Product Images Small, ni Polyvore.

> **Important — Secrets Colab**
>
> - Ton secret Kaggle s’appelle **`KAGGLE_API`**. Le notebook le lit sous ce nom, puis le transmet à Kaggle via la variable d’environnement requise.
> - `GITHUB_TOKEN` est facultatif et nécessaire seulement si ton dépôt GitHub est privé.


## 1. Monter Google Drive


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Cloner ou mettre à jour le repo GitHub

Le code du projet reste dans GitHub. Colab clone une copie temporaire dans `/content`.

- Si le repo est public : aucune information supplémentaire n’est nécessaire.
- S’il est privé : crée un Secret Colab optionnel `GITHUB_TOKEN` avec un token GitHub ayant seulement l’accès lecture au contenu du repo.


In [24]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

REPO_URL = "https://github.com/MilFhey/fit-outfit-advisor.git"
REPO_DIR = Path("/content/fit-outfit-advisor")
BRANCH = "main"


def get_optional_secret(name: str) -> str | None:
    try:
        value = userdata.get(name)
    except Exception:
        return None
    return value or None


def build_git_environment(github_token: str | None) -> tuple[dict[str, str], Path | None]:
    """Retourne un environnement Git ; utilise GIT_ASKPASS seulement si le repo est privé."""
    env = os.environ.copy()

    if not github_token:
        return env, None

    askpass_file = Path("/tmp/git_askpass.sh")
    askpass_file.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
        encoding="utf-8",
    )
    askpass_file.chmod(0o700)

    env["GITHUB_TOKEN"] = github_token
    env["GIT_ASKPASS"] = str(askpass_file)
    env["GIT_TERMINAL_PROMPT"] = "0"

    return env, askpass_file


github_token = get_optional_secret("GITHUB_TOKEN")
git_env, askpass_file = build_git_environment(github_token)

try:
    # Supprime uniquement un dossier incomplet provenant d’un clone interrompu.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        shutil.rmtree(REPO_DIR)

    if REPO_DIR.exists():
        print(f"Repo déjà présent, mise à jour : {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], env=git_env, check=True)
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
            env=git_env,
            check=True,
        )
    else:
        print(f"Clonage du repo : {REPO_URL}")
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            env=git_env,
            check=True,
        )
finally:
    if askpass_file is not None:
        askpass_file.unlink(missing_ok=True)

# Le repo peut contenir le projet directement à sa racine ou dans un dossier imbriqué.
candidate_project_dirs = [
    REPO_DIR,
    REPO_DIR / "fit-outfit-advisor",
]

PROJECT_DIR = next(
    (candidate for candidate in candidate_project_dirs if (candidate / "src").is_dir()),
    None,
)

if PROJECT_DIR is None:
    repo_contents = [path.name for path in REPO_DIR.iterdir()] if REPO_DIR.exists() else []
    raise FileNotFoundError(
        "Impossible de localiser le dossier src du projet. "
        f"Contenu de {REPO_DIR} : {repo_contents}"
    )

os.chdir(PROJECT_DIR)

print(f"Repo cloné : {REPO_DIR}")
print(f"Projet détecté : {PROJECT_DIR}")
print(f"Répertoire courant : {Path.cwd()}")


Repo déjà présent, mise à jour : /content/fit-outfit-advisor
Repo cloné : /content/fit-outfit-advisor
Projet détecté : /content/fit-outfit-advisor/fit-outfit-advisor
Répertoire courant : /content/fit-outfit-advisor/fit-outfit-advisor


## 3. Installer les dépendances


In [5]:
requirements_path = PROJECT_DIR / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(f"requirements.txt absent : {requirements_path}")

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_path)],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "kaggle"], check=True)

print("✅ Dépendances installées.")


✅ Dépendances installées.


## 4. Créer les dossiers temporaires Colab


In [6]:
RUNTIME_ROOT = Path("/content/fit-outfit-runtime")
KAGGLE_DOWNLOAD_DIR = RUNTIME_ROOT / "kaggle_downloads"
CONTENT_DATA_DIR = RUNTIME_ROOT / "data"
CONTENT_ARTIFACT_DIR = RUNTIME_ROOT / "artifacts"

for directory in [RUNTIME_ROOT, KAGGLE_DOWNLOAD_DIR, CONTENT_DATA_DIR, CONTENT_ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(directory)


/content/fit-outfit-runtime
/content/fit-outfit-runtime/kaggle_downloads
/content/fit-outfit-runtime/data
/content/fit-outfit-runtime/artifacts


## 5. Charger le secret Kaggle

Ton Secret Colab s’appelle **`KAGGLE_API`**.

Sa valeur doit être uniquement ton token Kaggle moderne, qui commence par `KGAT_`.

Le code le lit sous le nom `KAGGLE_API`, puis le place dans `KAGGLE_API_TOKEN` pour que la commande `kaggle` puisse s’authentifier.


In [7]:
try:
    kaggle_token = userdata.get("KAGGLE_API")
except Exception as exc:
    raise ValueError(
        "Secret Colab introuvable : crée ou autorise le Secret nommé exactement KAGGLE_API."
    ) from exc

if not kaggle_token or not kaggle_token.startswith("KGAT_"):
    raise ValueError(
        "Le Secret KAGGLE_API est absent ou invalide. "
        "Il doit contenir uniquement un token Kaggle moderne commençant par KGAT_."
    )

# Kaggle CLI attend cette variable d’environnement ; le Secret Colab peut conserver ton nom KAGGLE_API.
os.environ["KAGGLE_API_TOKEN"] = kaggle_token
os.environ["KAGGLE_API"] = kaggle_token

print("✅ Token Kaggle chargé depuis le Secret Colab KAGGLE_API.")


✅ Token Kaggle chargé depuis le Secret Colab KAGGLE_API.


## 6. Tester l’accès à Kaggle


In [8]:
result = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "clothing fit dataset"],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        "Échec d’authentification ou de connexion Kaggle.\n"
        f"Erreur : {result.stderr}"
    )

print(result.stdout[:2000])


ref                                                        title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------  -------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
rmisra/clothing-fit-dataset-for-size-recommendation        Clothing Fit Dataset for Size Recommendation         41600157  2018-08-21 19:00:16.793000          14209        201                1  
jocelyndumlao/consumer-review-of-clothing-product          Consumer Review of Clothing Product                   4335987  2023-10-19 04:45:23.063000           4543        114                1  
marquis03/high-resolution-viton-zalando-dataset            VITON-HD                                           4709595268  2023-10-13 12:03:51.400000          10796         57                1  
rmisra/news-category-dataset  

## 7. Télécharger le dataset ModCloth


In [9]:
KAGGLE_DATASET = "rmisra/clothing-fit-dataset-for-size-recommendation"
FORCE_DOWNLOAD = False

existing_files = [path for path in KAGGLE_DOWNLOAD_DIR.rglob("*") if path.is_file()]

if FORCE_DOWNLOAD or not existing_files:
    command = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        KAGGLE_DATASET,
        "-p",
        str(KAGGLE_DOWNLOAD_DIR),
        "--unzip",
    ]
    subprocess.run(command, check=True)
else:
    print("Téléchargement déjà présent : réutilisation des fichiers existants.")

downloaded_files = sorted(path for path in KAGGLE_DOWNLOAD_DIR.rglob("*") if path.is_file())

if not downloaded_files:
    raise FileNotFoundError(
        "Aucun fichier téléchargé depuis Kaggle. Vérifie le token et le slug du dataset."
    )

for path in downloaded_files:
    print(path)


/content/fit-outfit-runtime/kaggle_downloads/modcloth_final_data.json
/content/fit-outfit-runtime/kaggle_downloads/renttherunway_final_data.json


## 8. Détecter le fichier ModCloth

Le dataset peut être fourni en CSV, JSON ou JSONL.  
S’il est en JSON/JSONL, le notebook génère un CSV temporaire dans l’espace Colab.


In [10]:
import pandas as pd

csv_files = sorted(KAGGLE_DOWNLOAD_DIR.rglob("*.csv"))
modcloth_csv_files = [path for path in csv_files if "modcloth" in path.name.lower()]

if modcloth_csv_files:
    DATASET_PATH = modcloth_csv_files[0]
    print(f"CSV ModCloth détecté : {DATASET_PATH}")
elif csv_files:
    DATASET_PATH = csv_files[0]
    print(f"CSV détecté : {DATASET_PATH}")
else:
    json_files = sorted(
        [*KAGGLE_DOWNLOAD_DIR.rglob("*.json"), *KAGGLE_DOWNLOAD_DIR.rglob("*.jsonl")]
    )
    modcloth_json_files = [path for path in json_files if "modcloth" in path.name.lower()]

    if not modcloth_json_files:
        raise FileNotFoundError(
            "Aucun CSV, JSON ou JSONL ModCloth détecté dans le téléchargement Kaggle."
        )

    source_json = modcloth_json_files[0]
    print(f"JSON/JSONL ModCloth détecté : {source_json}")

    try:
        df_json = pd.read_json(source_json, lines=True)
    except ValueError:
        df_json = pd.read_json(source_json)

    DATASET_PATH = CONTENT_DATA_DIR / "modcloth_final_data.csv"
    df_json.to_csv(DATASET_PATH, index=False)
    print(f"CSV temporaire généré : {DATASET_PATH}")

print(f"DATASET_PATH = {DATASET_PATH}")


JSON/JSONL ModCloth détecté : /content/fit-outfit-runtime/kaggle_downloads/modcloth_final_data.json
CSV temporaire généré : /content/fit-outfit-runtime/data/modcloth_final_data.csv
DATASET_PATH = /content/fit-outfit-runtime/data/modcloth_final_data.csv


## 9. Inspecter le dataset avant entraînement


In [11]:
df = pd.read_csv(DATASET_PATH, low_memory=False)

print("df.shape =", df.shape)

print("\nColonnes :")
print(list(df.columns))

print("\nTypes :")
display(df.dtypes.to_frame("dtype"))

print("\nAperçu :")
display(df.head())

missing_values = df.isna().sum().sort_values(ascending=False)
print("\nValeurs manquantes par colonne :")
display(missing_values.to_frame("missing_count"))


df.shape = (82790, 18)

Colonnes :
['item_id', 'waist', 'size', 'quality', 'cup size', 'hips', 'bra size', 'category', 'bust', 'height', 'user_name', 'length', 'fit', 'user_id', 'shoe size', 'shoe width', 'review_summary', 'review_text']

Types :


,dtype
item_id,int64
waist,float64
size,int64
quality,float64
cup size,object
hips,float64
bra size,float64
category,object
bust,object
height,object



Aperçu :


,item_id,waist,size,quality,cup size,hips,bra size,category,bust,height,user_name,length,fit,user_id,shoe size,shoe width,review_summary,review_text
0,123373,29.0,7,5.0,d,38.0,34.0,new,36,5ft 6in,Emily,just right,small,991571,NaN,NaN,NaN,NaN
1,123373,31.0,13,3.0,b,30.0,36.0,new,NaN,5ft 2in,sydneybraden2001,just right,small,587883,NaN,NaN,NaN,NaN
2,123373,30.0,7,2.0,b,NaN,32.0,new,NaN,5ft 7in,Ugggh,slightly long,small,395665,9.0,NaN,NaN,NaN
3,123373,NaN,21,5.0,dd/e,NaN,NaN,new,NaN,NaN,alexmeyer626,just right,fit,875643,NaN,NaN,NaN,NaN
4,123373,NaN,18,5.0,b,NaN,36.0,new,NaN,5ft 2in,dberrones1,slightly long,small,944840,NaN,NaN,NaN,NaN



Valeurs manquantes par colonne :


,missing_count
waist,79908
bust,70936
shoe width,64183
shoe size,54875
hips,26726
review_summary,6732
review_text,6732
cup size,6255
bra size,6018
height,1107


In [12]:
TOTAL = len(df)

missing = df.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / TOTAL * 100).round(2)
display(missing.sort_values("missing_count", ascending=False))

print("Distribution fit")
display(df["fit"].value_counts(dropna=False))
display((df["fit"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("Distribution category")
display(df["category"].value_counts(dropna=False))
display((df["category"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("Fit par category")
display(pd.crosstab(df["category"], df["fit"]))
display(pd.crosstab(df["category"], df["fit"], normalize="index").round(3))

,missing_count,missing_pct
waist,79908,96.52
bust,70936,85.68
shoe width,64183,77.53
shoe size,54875,66.28
hips,26726,32.28
review_summary,6732,8.13
review_text,6732,8.13
cup size,6255,7.56
bra size,6018,7.27
height,1107,1.34


Distribution fit


,count
fit,
fit,56757
large,13059
small,12974


,proportion
fit,
fit,68.56
large,15.77
small,15.67


Distribution category


,count
category,
new,21488
tops,20364
dresses,18650
bottoms,15266
outerwear,4223
sale,2524
wedding,275


,proportion
category,
new,25.95
tops,24.60
dresses,22.53
bottoms,18.44
outerwear,5.10
sale,3.05
wedding,0.33


Fit par category


fit,fit,large,small
category,,,
bottoms,10660,2058,2548
dresses,13574,2562,2514
new,14423,2978,4087
outerwear,2793,816,614
sale,1590,434,500
tops,13498,4168,2698
wedding,219,43,13


fit,fit,large,small
category,,,
bottoms,0.698,0.135,0.167
dresses,0.728,0.137,0.135
new,0.671,0.139,0.190
outerwear,0.661,0.193,0.145
sale,0.630,0.172,0.198
tops,0.663,0.205,0.132
wedding,0.796,0.156,0.047


In [13]:
explicit_categories = ["tops", "dresses", "bottoms", "outerwear", "wedding"]
ambiguous_categories = ["new", "sale"]

df_no_commercial = df[~df["category"].isin(ambiguous_categories)].copy()
df_explicit = df[df["category"].isin(explicit_categories)].copy()

print("Dataset complet:", df.shape)
print("Sans new/sale:", df_no_commercial.shape)
print("Categories explicites seulement:", df_explicit.shape)

print("Fit complet")
display(df["fit"].value_counts(normalize=True).round(3))

print("Fit sans new/sale")
display(df_no_commercial["fit"].value_counts(normalize=True).round(3))

print("Fit categories explicites")
display(df_explicit["fit"].value_counts(normalize=True).round(3))

Dataset complet: (82790, 18)
Sans new/sale: (58778, 18)
Categories explicites seulement: (58778, 18)
Fit complet


,proportion
fit,
fit,0.686
large,0.158
small,0.157


Fit sans new/sale


,proportion
fit,
fit,0.693
large,0.164
small,0.143


Fit categories explicites


,proportion
fit,
fit,0.693
large,0.164
small,0.143


In [14]:
print("size describe")
display(df["size"].describe())

print("size par fit")
display(df.groupby("fit")["size"].describe())

print("size par category")
display(df.groupby("category")["size"].describe())

print("Exemples size/category/fit")
display(df[["size", "category", "fit", "height", "hips", "bra size", "cup size"]].head(30))

size describe


,size
count,82790.000000
mean,12.661602
std,8.271952
min,0.000000
25%,8.000000
50%,12.000000
75%,15.000000
max,38.000000


size par fit


,count,mean,std,min,25%,50%,75%,max
fit,,,,,,,,
fit,56757.0,12.036718,7.896294,0.0,8.0,12.0,15.0,38.0
large,13059.0,14.238839,9.272834,0.0,8.0,12.0,20.0,38.0
small,12974.0,13.807692,8.473869,0.0,8.0,12.0,15.0,38.0


size par category


,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
bottoms,15266.0,12.437246,8.078311,0.0,8.0,12.0,15.0,38.0
dresses,18650.0,11.966381,7.777325,0.0,7.0,12.0,15.0,38.0
new,21488.0,12.886681,8.389745,0.0,8.0,12.0,15.0,38.0
outerwear,4223.0,12.616386,7.892644,1.0,8.0,12.0,15.0,38.0
sale,2524.0,12.963550,8.404193,0.0,8.0,12.0,15.0,38.0
tops,20364.0,13.165734,8.716828,0.0,8.0,12.0,20.0,38.0
wedding,275.0,15.269091,8.884516,1.0,8.0,12.0,20.0,38.0


Exemples size/category/fit


,size,category,fit,height,hips,bra size,cup size
0,7,new,small,5ft 6in,38.0,34.0,d
1,13,new,small,5ft 2in,30.0,36.0,b
2,7,new,small,5ft 7in,NaN,32.0,b
3,21,new,fit,NaN,NaN,NaN,dd/e
4,18,new,small,5ft 2in,NaN,36.0,b
5,11,new,small,5ft 4in,41.0,36.0,c
6,5,new,large,5ft 3in,NaN,32.0,b
7,11,new,small,5ft 5in,42.0,38.0,d
8,30,new,small,5ft 10in,50.0,42.0,d
9,13,new,fit,5ft 6in,41.0,36.0,dd/e


In [15]:
measurement_cols = ["height", "hips", "bra size", "cup size", "waist", "bust"]

for col in measurement_cols:
    print(f"\n=== {col} ===")
    print("missing:", df[col].isna().sum(), f"({df[col].isna().mean() * 100:.2f}%)")
    display(df[col].value_counts(dropna=False).head(30))

print("\nMensurations manquantes par fit")
for col in measurement_cols:
    missing_by_fit = df.assign(is_missing=df[col].isna()).groupby("fit")["is_missing"].mean().round(3)
    print(f"\n{col}")
    display(missing_by_fit)

print("\nMensurations numériques par fit")
for col in ["hips", "bra size", "waist"]:
    print(f"\n{col}")
    display(df.groupby("fit")[col].describe())

print("\nCup size par fit")
display(pd.crosstab(df["cup size"], df["fit"], normalize="index").round(3))


=== height ===
missing: 1107 (1.34%)


,count
height,
5ft 4in,11928
5ft 6in,11891
5ft 5in,9418
5ft 7in,9161
5ft 3in,8680
5ft 2in,7684
5ft 8in,6420
5ft 9in,4574
5ft 1in,3571



=== hips ===
missing: 26726 (32.28%)


,count
hips,
NaN,26726
35.0,6090
40.0,5452
38.0,4961
36.0,4829
42.0,3637
39.0,3263
37.0,3162
41.0,3146



=== bra size ===
missing: 6018 (7.27%)


...

,count,mean,std,min,25%,50%,75%,max
fit,,,,,,,,
fit,52601.0,35.800232,3.147661,28.0,34.0,36.0,38.0,48.0
large,12141.0,36.266864,3.411476,28.0,34.0,36.0,38.0,48.0
small,12030.0,36.426268,3.298719,28.0,34.0,36.0,38.0,48.0



waist


,count,mean,std,min,25%,50%,75%,max
fit,,,,,,,,
fit,1725.0,31.059710,5.227141,20.0,28.0,30.0,34.00,50.0
large,606.0,31.892739,5.410737,20.0,28.0,31.0,35.75,50.0
small,551.0,31.500907,5.370585,22.0,28.0,30.0,34.00,50.0



Cup size par fit


fit,fit,large,small
cup size,,,
a,0.726,0.157,0.117
aa,0.726,0.159,0.115
b,0.720,0.142,0.138
c,0.690,0.152,0.158
d,0.681,0.155,0.164
dd/e,0.669,0.164,0.167
ddd/f,0.637,0.184,0.179
dddd/g,0.625,0.194,0.181
h,0.606,0.208,0.186


In [16]:
# Parse simple de height en cm pour analyse descriptive
import re
import numpy as np
import pandas as pd

def parse_height_to_cm(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    nums = re.findall(r"\d+(?:\.\d+)?", text)
    if not nums:
        return np.nan
    if "ft" in text:
        feet = float(nums[0])
        inches = float(nums[1]) if len(nums) > 1 else 0
        return round((feet * 12 + inches) * 2.54, 2)
    if "cm" in text:
        return float(nums[0])
    return np.nan

analysis = df.copy()
analysis["height_cm"] = analysis["height"].map(parse_height_to_cm)

print("height_cm par fit")
display(analysis.groupby("fit")["height_cm"].describe())

print("size par fit")
display(analysis.groupby("fit")["size"].describe())

print("size par category et fit")
display(analysis.groupby(["category", "fit"])["size"].describe())

print("Correlation numérique simple")
numeric_cols = ["size", "height_cm", "hips", "bra size"]
display(analysis[numeric_cols].corr(numeric_only=True))

height_cm par fit


,count,mean,std,min,25%,50%,75%,max
fit,,,,,,,,
fit,56008.0,165.347297,7.209751,91.44,160.02,165.1,170.18,241.30
large,12870.0,165.513860,7.231408,91.44,160.02,165.1,170.18,231.14
small,12805.0,165.974768,7.391252,91.44,160.02,165.1,170.18,241.30


size par fit


,count,mean,std,min,25%,50%,75%,max
fit,,,,,,,,
fit,56757.0,12.036718,7.896294,0.0,8.0,12.0,15.0,38.0
large,13059.0,14.238839,9.272834,0.0,8.0,12.0,20.0,38.0
small,12974.0,13.807692,8.473869,0.0,8.0,12.0,15.0,38.0


size par category et fit


count       mean       std  min  25%   50%   75%   max
category  fit                                                            
bottoms   fit    10660.0  11.759662  7.581203  0.0  8.0  10.0  15.0  38.0
          large   2058.0  13.268707  9.090973  1.0  8.0  12.0  20.0  38.0
          small   2548.0  14.600471  8.756965  0.0  8.0  12.0  20.0  38.0
dresses   fit    13574.0  11.278695  7.321463  0.0  7.0   9.0  15.0  38.0
          large   2562.0  14.607728  9.087556  0.0  8.0  12.0  20.0  38.0
          small   2514.0  12.987669  8.030003  0.0  8.0  12.0  15.0  38.0
new       fit    14423.0  12.358663  8.151184  0.0  8.0  12.0  15.0  38.0
          large   2978.0  15.099731  9.624446  0.0  8.0  12.0  20.0  38.0
          small   4087.0  13.137509  7.970466  0.0  8.0  12.0  15.0  38.0
outerwear fit     2793.0  11.687791  7.319161  1.0  8.0  12.0  15.0  38.0
          large    816.0  16.397059  9.437125  1.0  8.0  15.0  26.0  38.0
          small    614.0  11.815961  6.561070  1.0  8.0  12.0  12.0  38.0
sale      fit     1590.0  12.125786  8.035355  0.0  8.0  12.0  15.0  38.0
          large    434.0  15.237327  9.278667  1.0  8.0  15.0  20.0  38.0
          small    500.0  13.654000  8.342270  1.0  8.0  12.0  15.0  38.0
tops      fit    13498.0  12.680397  8.403240  0.0  8.0  12.0  15.0  38.0
          large   4168.0  13.348369  9.028714  0.0  8.0  12.0  20.0  38.0
          small   2698.0  15.311712  9.408397  1.0  8.0  12.0  20.0  38.0
wedding   fit      219.0  15.433790  8.719847  3.0  8.0  12.0  20.0  38.0
          large     43.0  14.348837  9.712250  1.0  4.0  12.0  22.0  32.0
          small     13.0  15.538462  9.341800  4.0  8.0  12.0  20.0  38.0

Correlation numérique simple


,size,height_cm,hips,bra size
size,1.000000,0.206594,0.747257,0.788460
height_cm,0.206594,1.000000,0.191092,0.179168
hips,0.747257,0.191092,1.000000,0.671034
bra size,0.788460,0.179168,0.671034,1.000000


In [17]:
candidate_features = ["size", "category", "height", "hips", "bra size", "cup size", "fit"]

subsets = {
    "minimal_size_category_height": ["size", "category", "height", "fit"],
    "with_hips": ["size", "category", "height", "hips", "fit"],
    "with_bra_cup": ["size", "category", "height", "bra size", "cup size", "fit"],
    "with_hips_bra_cup": ["size", "category", "height", "hips", "bra size", "cup size", "fit"],
}

for name, cols in subsets.items():
    temp = df[cols].dropna()
    print(f"\n=== {name} ===")
    print("rows:", len(temp), f"({len(temp) / len(df) * 100:.2f}%)")
    print("fit distribution")
    display(temp["fit"].value_counts(normalize=True).round(3))
    print("category distribution")
    display(temp["category"].value_counts(normalize=True).round(3))


=== minimal_size_category_height ===
rows: 81683 (98.66%)
fit distribution


,proportion
fit,
fit,0.686
large,0.158
small,0.157


category distribution


,proportion
category,
new,0.259
tops,0.246
dresses,0.226
bottoms,0.184
outerwear,0.051
sale,0.030
wedding,0.003



=== with_hips ===
rows: 55828 (67.43%)
fit distribution


...

,proportion
category,
new,0.262
tops,0.250
dresses,0.228
bottoms,0.177
outerwear,0.052
sale,0.029
wedding,0.003



=== with_hips_bra_cup ===
rows: 54345 (65.64%)
fit distribution


,proportion
fit,
fit,0.687
large,0.161
small,0.151


category distribution


,proportion
category,
new,0.261
tops,0.247
dresses,0.221
bottoms,0.189
outerwear,0.051
sale,0.028
wedding,0.003


In [18]:
for col in ["size", "category", "cup size", "bra size", "hips", "height"]:
    print(f"\n=== {col} ===")
    print("nunique:", df[col].nunique(dropna=False))
    display(df[col].value_counts(dropna=False).head(50))


=== size ===
nunique: 29


,count
size,
8,17893
12,17343
4,13883
20,7292
15,6883
26,5656
32,3613
1,1784
38,1461



=== category ===
nunique: 7


,count
category,
new,21488
tops,20364
dresses,18650
bottoms,15266
outerwear,4223
sale,2524
wedding,275



=== cup size ===
nunique: 13


...

,count
bra size,
34.0,22412
36.0,19624
38.0,11923
32.0,10026
NaN,6018
40.0,5115
42.0,3373
44.0,2014
30.0,1052



=== hips ===
nunique: 32


,count
hips,
NaN,26726
35.0,6090
40.0,5452
38.0,4961
36.0,4829
42.0,3637
39.0,3263
37.0,3162
41.0,3146



=== height ===
nunique: 42


,count
height,
5ft 4in,11928
5ft 6in,11891
5ft 5in,9418
5ft 7in,9161
5ft 3in,8680
5ft 2in,7684
5ft 8in,6420
5ft 9in,4574
5ft 1in,3571


In [19]:
analysis = df.copy()
analysis["height_cm"] = analysis["height"].map(parse_height_to_cm)

print("height_cm outliers bas")
display(analysis[analysis["height_cm"] < 130][["height", "height_cm", "fit", "category", "size"]].head(50))

print("height_cm outliers hauts")
display(analysis[analysis["height_cm"] > 210][["height", "height_cm", "fit", "category", "size"]].head(50))

print("Nombre outliers height")
print("height_cm < 130:", (analysis["height_cm"] < 130).sum())
print("height_cm > 210:", (analysis["height_cm"] > 210).sum())

height_cm outliers bas


,height,height_cm,fit,category,size
468,3ft 4in,101.60,fit,new,5
3484,3ft,91.44,fit,new,4
4952,3ft,91.44,fit,new,32
5661,3ft,91.44,fit,new,12
9349,4ft 2in,127.00,fit,dresses,11
10573,3ft,91.44,fit,dresses,32
11427,3ft 4in,101.60,fit,dresses,8
13601,3ft 11in,119.38,small,dresses,4
20064,3ft,91.44,fit,dresses,32
20483,3ft,91.44,fit,dresses,15


height_cm outliers hauts


,height,height_cm,fit,category,size
9080,7ft 11in,241.30,fit,dresses,12
10369,7ft 11in,241.30,small,dresses,8
11619,7ft 11in,241.30,small,dresses,26
19094,7ft 6in,228.60,fit,dresses,18
19586,7ft 11in,241.30,small,dresses,13
29098,7ft 11in,241.30,fit,new,12
35763,7ft 5in,226.06,fit,new,12
37635,7ft 11in,241.30,small,new,12
38975,7ft 11in,241.30,fit,tops,12
39588,7ft 11in,241.30,small,tops,12


Nombre outliers height
height_cm < 130: 28
height_cm > 210: 29


In [20]:
for col in ["size", "hips", "bra size"]:
    print(f"\n=== {col} ===")
    display(df[col].describe())
    display(df[[col, "fit", "category"]].sort_values(col).head(20))
    display(df[[col, "fit", "category"]].sort_values(col, ascending=False).head(20))


=== size ===


,size
count,82790.000000
mean,12.661602
std,8.271952
min,0.000000
25%,8.000000
50%,12.000000
75%,15.000000
max,38.000000


,size,fit,category
44475,0,large,tops
3517,0,large,new
77496,0,fit,new
78258,0,fit,new
47151,0,fit,tops
47158,0,fit,sale
1746,0,small,new
53078,0,fit,tops
53004,0,fit,tops
53626,0,fit,tops


,size,fit,category
66479,38,large,bottoms
44263,38,fit,tops
53705,38,fit,tops
66432,38,fit,bottoms
66440,38,fit,bottoms
66454,38,small,bottoms
44251,38,large,tops
32176,38,fit,new
53788,38,small,tops
53807,38,large,tops



=== hips ===


...

,hips,fit,category
29406,60.0,fit,new
54075,60.0,fit,tops
54094,60.0,fit,tops
2470,60.0,large,new
59445,60.0,large,bottoms
7469,60.0,small,new
66685,60.0,fit,bottoms
74346,60.0,large,bottoms
6582,60.0,large,new
44396,60.0,small,tops



=== bra size ===


,bra size
count,76772.000000
mean,35.972125
std,3.224907
min,28.000000
25%,34.000000
50%,36.000000
75%,38.000000
max,48.000000


,bra size,fit,category
12728,28.0,fit,dresses
77193,28.0,small,bottoms
12027,28.0,fit,dresses
12395,28.0,large,dresses
51401,28.0,fit,tops
20358,28.0,fit,dresses
20225,28.0,fit,dresses
20265,28.0,fit,dresses
9942,28.0,small,dresses
19841,28.0,large,sale


,bra size,fit,category
70373,48.0,large,bottoms
70338,48.0,small,bottoms
41489,48.0,large,tops
70254,48.0,fit,bottoms
70309,48.0,fit,bottoms
21209,48.0,small,dresses
58223,48.0,fit,tops
21370,48.0,small,dresses
21432,48.0,small,dresses
58011,48.0,fit,tops


In [21]:
explicit_categories = ["tops", "dresses", "bottoms", "outerwear", "wedding"]
ambiguous_categories = ["new", "sale"]

print("Catégories inconnues")
print(set(df["category"].dropna().unique()) - set(explicit_categories) - set(ambiguous_categories))

print("Counts catégories explicites")
display(df[df["category"].isin(explicit_categories)]["category"].value_counts())

print("Counts new/sale")
display(df[df["category"].isin(ambiguous_categories)]["category"].value_counts())

Catégories inconnues
set()
Counts catégories explicites


,count
category,
tops,20364
dresses,18650
bottoms,15266
outerwear,4223
wedding,275


Counts new/sale


,count
category,
new,21488
sale,2524


## 10. Lancer l’entraînement ModCloth V3

Cette cellule lance le vrai script du repo.  
Le script doit créer les artefacts versionnés dans `models/fit_v3/`.

L’entraînement ne démarre que si :
- le dataset est présent ;
- le dossier projet est détecté ;
- `src/training/train_fit_model.py` existe.


In [28]:
EPOCHS = 30
BATCH_SIZE = 64

training_script = PROJECT_DIR / "src" / "training" / "train_fit_model_v3.py"

if not Path(DATASET_PATH).exists():
    raise FileNotFoundError(f"Dataset absent : {DATASET_PATH}")

if not training_script.exists():
    raise FileNotFoundError(f"Script d’entraînement absent : {training_script}")

training_env = os.environ.copy()
training_env["PYTHONPATH"] = (
    f"{PROJECT_DIR}{os.pathsep}{training_env.get('PYTHONPATH', '')}".rstrip(os.pathsep)
)

print(f"Répertoire courant : {PROJECT_DIR}")
print(f"Script exécuté : {training_script}")
print(f"Dataset : {DATASET_PATH}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "src.training.train_fit_model_v3",
        "--dataset",
        str(DATASET_PATH),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
    ],
    cwd=str(PROJECT_DIR),
    env=training_env,
    check=True,
)

Répertoire courant : /content/fit-outfit-advisor/fit-outfit-advisor
Script exécuté : /content/fit-outfit-advisor/fit-outfit-advisor/src/training/train_fit_model_v3.py
Dataset : /content/fit-outfit-runtime/data/modcloth_final_data.csv


CompletedProcess(args=['/usr/bin/python3', '-m', 'src.training.train_fit_model_v3', '--dataset', '/content/fit-outfit-runtime/data/modcloth_final_data.csv', '--epochs', '30', '--batch-size', '64'], returncode=0)

In [31]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "src.training.train_fit_model_v3",
        "--dataset",
        str(DATASET_PATH),
        "--category-scope",
        "explicit",
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
    ],
    cwd=str(PROJECT_DIR),
    env=training_env,
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'src.training.train_fit_model_v3', '--dataset', '/content/fit-outfit-runtime/data/modcloth_final_data.csv', '--category-scope', 'explicit', '--epochs', '30', '--batch-size', '64'], returncode=0)

## 11. Vérifier les métriques et artefacts V3


In [29]:
import json
from pathlib import Path

FIT_V3_DIR = PROJECT_DIR / "models" / "fit_v3"

artifact_paths = [
    FIT_V3_DIR / "fit_model.keras",
    FIT_V3_DIR / "fit_estimator.joblib",
    FIT_V3_DIR / "fit_preprocessor.joblib",
    FIT_V3_DIR / "fit_label_encoder.joblib",
    FIT_V3_DIR / "metadata.json",
    FIT_V3_DIR / "metrics.json",
    FIT_V3_DIR / "confusion_matrix_raw.png",
    FIT_V3_DIR / "confusion_matrix_normalized.png",
    FIT_V3_DIR / "training_history.png",
]

print("Artefacts V3 :")
for path in artifact_paths:
    if path.exists():
        print("OK     ", path.name, path.stat().st_size, "bytes")
    else:
        print("ABSENT ", path.name)

metadata = json.loads((FIT_V3_DIR / "metadata.json").read_text(encoding="utf-8"))
metrics = json.loads((FIT_V3_DIR / "metrics.json").read_text(encoding="utf-8"))

print("\n=== Metadata ===")
print("version:", metadata.get("version"))
print("category_scope:", metadata.get("category_scope"))
print("selected_experiment:", metadata.get("selected_experiment"))
print("selected_model_type:", metadata.get("selected_model_type"))
print("model_status:", metadata.get("model_status"))
print("promotable_to_streamlit:", metadata.get("promotable_to_streamlit"))
print("reason:", metadata.get("reason_for_selection"))

print("\n=== Validation metrics ===")
for name, result in metrics["validation_metrics"].items():
    print(name, {
        "accuracy": round(result["accuracy"], 4),
        "balanced_accuracy": round(result["balanced_accuracy"], 4),
        "macro_f1": round(result["macro_f1"], 4),
        "weighted_f1": round(result["weighted_f1"], 4),
    })

print("\n=== Test metrics selected ===")
selected_test = metrics["test_metrics"]["selected_experiment"]
print({
    "accuracy": round(selected_test["accuracy"], 4),
    "balanced_accuracy": round(selected_test["balanced_accuracy"], 4),
    "macro_f1": round(selected_test["macro_f1"], 4),
    "weighted_f1": round(selected_test["weighted_f1"], 4),
})

print("\n=== Per class test ===")
for label in metadata["class_labels"]:
    cls = selected_test["per_class"][label]
    print(label, {
        "precision": round(cls["precision"], 4),
        "recall": round(cls["recall"], 4),
        "f1": round(cls["f1-score"], 4),
        "support": int(cls["support"]),
    })

Artefacts V3 :
OK      fit_model.keras 120916 bytes
ABSENT  fit_estimator.joblib
OK      fit_preprocessor.joblib 4319 bytes
OK      fit_label_encoder.joblib 495 bytes
OK      metadata.json 7564 bytes
OK      metrics.json 93216 bytes
OK      confusion_matrix_raw.png 50783 bytes
OK      confusion_matrix_normalized.png 50762 bytes
OK      training_history.png 94851 bytes

=== Metadata ===
version: fit_v3
category_scope: all
selected_experiment: mlp_class_weight
selected_model_type: keras_mlp
model_status: experimental_only
promotable_to_streamlit: False
reason: mlp_class_weight selected on validation only: macro_f1=0.3679 vs majority_baseline=0.2712, balanced_accuracy=0.4287 vs majority_baseline=0.3333, small_recall=0.3494, large_recall=0.5372.

=== Validation metrics ===
majority_baseline {'accuracy': 0.6858, 'balanced_accuracy': 0.3333, 'macro_f1': 0.2712, 'weighted_f1': 0.558}
logistic_regression {'accuracy': 0.6858, 'balanced_accuracy': 0.3333, 'macro_f1': 0.2712, 'weighted_f1': 0.558

## 12. Copier les artefacts V2 vers Google Drive


In [32]:
DRIVE_V3_EXPLICIT_DIR = Path("/content/drive/MyDrive/fit-outfit-advisor/artifacts/modcloth_fit_v3_explicit")
DRIVE_V3_EXPLICIT_DIR.mkdir(parents=True, exist_ok=True)

for path in FIT_V3_DIR.iterdir():
    if path.is_file():
        shutil.copy2(path, DRIVE_V3_EXPLICIT_DIR / path.name)
        print("copied", path.name)

print("Saved explicit run to:", DRIVE_V3_EXPLICIT_DIR)

copied metadata.json
copied .gitkeep
copied fit_model.keras
copied confusion_matrix_raw.png
copied metrics.json
copied fit_label_encoder.joblib
copied confusion_matrix_normalized.png
copied training_history.png
copied fit_preprocessor.joblib
Saved explicit run to: /content/drive/MyDrive/fit-outfit-advisor/artifacts/modcloth_fit_v3_explicit
